# Google Brain Ventilator Pressure Prediction - LightGBM ベースライン

**目的**: 人工呼吸器の吸気相における肺内圧（pressure）を予測する。

**ノートブックの流れ**（`google-brain-lightgbm-optuna` と同様の構成）  
分析設計 → データ前処理 → EDA → 特徴量エンジニアリング → データセット作成 → Optuna によるハイパーパラメータ探索（参考コード・コメントアウト）→ モデル学習（LightGBM / `LGBMRegressor`）→ 評価 → 提出用ファイル作成

---
## 1. 分析設計

コンペで何を予測し、どの指標で評価するかを明確にします。

| 項目 | 内容 | 選んだ根拠 |
|------|------|------------|
| **目的変数** | `pressure`（気道圧） | コンペの公式説明および Data タブの説明で「predict the pressure in the respiratory circuit」とされているため。 |
| **目的変数の種類** | **連続値（回帰）** | pressure は実数値（例: -1.895 ~ 64.0 程度）で、離散ラベルではなく連続的な物理量であるため。2値・カテゴリではない。 |
| **評価指標** | **MAE（Mean Absolute Error）** | コンペの「Evaluation」で MAE が指定されているため。予測圧と正解圧の絶対誤差の平均を最小化する。 |

**タスクの整理**  
- 時系列データ: 1 呼吸（breath）あたり 80 ステップ。各ステップで `time_step`, `u_in`, `u_out` が与えられ、そのステップの `pressure` を予測する。
- テストでは `pressure` が欠けているため、同じ形式の特徴量から pressure を予測して提出する。

---
## 2. ライブラリの読み込みとパス設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
# import optuna  # セクション7のパラメータ探索を使うときに有効化
import warnings

warnings.filterwarnings("ignore")

# Kaggle 環境専用パス（コンペの Data を追加したときのパス）
INPUT_DIR = "/kaggle/input/ventilator-pressure-prediction"
OUTPUT_DIR = "/kaggle/working"
TRAIN_PATH = f"{INPUT_DIR}/train.csv"
TEST_PATH = f"{INPUT_DIR}/test.csv"
SAMPLE_SUBMISSION_PATH = f"{INPUT_DIR}/sample_submission.csv"
SUBMISSION_PATH = f"{OUTPUT_DIR}/submission.csv"

TARGET = "pressure"
ID_COL = "id"

pd.set_option("display.max_columns", 20)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")
%matplotlib inline

---
## 3. データ前処理

データの読み込み、型・欠損の確認、breath 単位でソートして時系列の並びをそろえます。

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("【訓練データ】 行数:", len(train), ", 列数:", len(train.columns))
print("【テストデータ】 行数:", len(test), ", 列数:", len(test.columns))
print()
print("訓練データ 先頭5行:")
train.head()

In [ ]:
train.info()
print()
print("欠損値:")
print(train.isnull().sum())

In [ ]:
# breath_id と time_step でソート（時系列の順序を保つ）
train = train.sort_values(["breath_id", "time_step"]).reset_index(drop=True)
test = test.sort_values(["breath_id", "time_step"]).reset_index(drop=True)
print("breath_id, time_step でソート済み")

---
## 4. EDA（探索的データ分析）

`google-brain-lightgbm-optuna` と同様に、要約統計・カテゴリ（`R`, `C`）の件数・主要変数の分布・テスト先頭行を確認します。

In [ ]:
train.describe()

In [ ]:
print("欠損値（訓練）:")
print(train.isnull().sum())
print()
print("breath_id のユニーク数:", train["breath_id"].nunique())
print("各 breath のステップ数（先頭数件）:")
print(train["breath_id"].value_counts().head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=train, x="R", ax=axes[0], color="steelblue")
axes[0].set_title("R の件数")
sns.countplot(data=train, x="C", ax=axes[1], color="coral")
axes[1].set_title("C の件数")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
sns.histplot(train["u_in"], bins=10, kde=False, color="blue", ax=axes[0])
axes[0].set_title("u_in")
sns.histplot(train["u_out"], bins=10, kde=False, color="blue", ax=axes[1])
axes[1].set_title("u_out")
sns.histplot(train[TARGET], bins=10, kde=False, color="blue", ax=axes[2])
axes[2].set_title(TARGET)
plt.tight_layout()
plt.show()

In [ ]:
print("テストデータ 先頭5行:")
test.head()

---
## 5. 特徴量エンジニアリング

`google-brain-lightgbm-optuna` と同様に、**同一 breath 内**の `u_in` について累積和（`u_in_cumsum`）と 2 ステップラグ（`u_in_lag`）を追加します。  
※ 元カーネルは全体に対する `shift` ですが、呼吸をまたがないよう `breath_id` ごとに計算しています。

In [ ]:
train["u_in_cumsum"] = train.groupby("breath_id")["u_in"].cumsum()
test["u_in_cumsum"] = test.groupby("breath_id")["u_in"].cumsum()

train["u_in_lag"] = train.groupby("breath_id")["u_in"].shift(2).fillna(0)
test["u_in_lag"] = test.groupby("breath_id")["u_in"].shift(2).fillna(0)

print("特徴量追加後の列（訓練）:", list(train.columns))

---
## 6. データセット作成

`id`, `breath_id`, `pressure` を除いた特徴量を `X` とし、目的変数を `y` とします（`u_out` は特徴量に含めます）。

In [ ]:
X = train.drop(["id", "breath_id", "pressure"], axis=1)
y = train[TARGET]
X_test = test.drop(["id", "breath_id"], axis=1)

print("特徴量列:", list(X.columns))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)

---
## 7. ハイパーパラメータ探索（Optuna）

（参考）`train_test_split` で切り分けた検証 MAE を最小化するように `LGBMRegressor` のハイパーパラメータを探索するコード例です。  
現状は**下のセルはコメントアウト済み**です。`import optuna` を有効化し、セル内のコメントを外して `study.optimize` を実行してください。

In [ ]:
# def objective(trial):
#     X_train, X_valid, y_train, y_valid = train_test_split(
#         X, y, train_size=0.8, test_size=0.2, random_state=0
#     )
#     params = {
#         "objective": "regression",
#         "metric": "mae",
#         "boosting_type": "gbdt",
#         "n_estimators": 1000,
#         "random_state": 42,
#         "learning_rate": trial.suggest_categorical(
#             "learning_rate", [0.006, 0.008, 0.01, 0.014, 0.017, 0.02]
#         ),
#         "subsample": trial.suggest_float("subsample", 0.4, 1.0, log=True),
#         "subsample_freq": trial.suggest_float("subsample_freq", 0.4, 1.0, log=True),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
#         "min_child_weight": trial.suggest_int("min_child_weight", 5, 256),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
#         "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
#         "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
#     }
#     model = lgb.LGBMRegressor(**params)
#     model.fit(X_train, y_train)
#     preds = model.predict(X_valid)
#     return mean_absolute_error(y_valid, preds)
#
#
# # study = optuna.create_study(direction="minimize")
# # study.optimize(objective, n_trials=10)
# # print("Number of finished trials:", len(study.trials))
# # print("Best trial:", study.best_trial.params)

---
## 8. モデル学習（LightGBM）

`train_test_split` で学習・検証に分割し、`LGBMRegressor` を学習します。ハイパーパラメータは `google-brain-lightgbm-optuna` で示されている **探索後の一例**（カーネル内の固定値）をそのまま利用します。独自に Optuna を回した場合は `study.best_trial.params` を反映してください。

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

lgb_params = {
    "objective": "regression",
    "metric": "mae",
    "boosting_type": "gbdt",
    "n_estimators": 1000,
    "random_state": 42,
    "learning_rate": 0.017,
    "subsample": 0.6706735076307812,
    "subsample_freq": 0.9731836936473381,
    "colsample_bytree": 0.7981147731267384,
    "reg_alpha": 0.29250836566881794,
    "reg_lambda": 0.0032438602599939702,
    "min_child_weight": 134,
    "min_child_samples": 26,
    "bagging_fraction": 0.6263245217964235,
    "bagging_freq": 1,
}

model = lgb.LGBMRegressor(**lgb_params)
model.fit(X_train, y_train)

In [ ]:
pred_valid = model.predict(X_valid)
print("検証データ MAE:", mean_absolute_error(y_valid, pred_valid))

---
## 9. 提出用ファイルの作成

`sample_submission.csv` の行順を保ったまま `pressure` を上書きして保存します（`google-brain-lightgbm-optuna` と同じ手順）。

In [ ]:
preds = model.predict(X_test)
pred_by_id = pd.Series(preds, index=test[ID_COL])
submission[TARGET] = submission[ID_COL].map(pred_by_id)
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"提出用ファイルを保存しました: {SUBMISSION_PATH}")
submission.head(10)